# Citadel Colab TPU launcher (thin — first execution surface)
All logic lives in `citadel_tpu/`. This notebook only: setup → probe → T0 → STOP unless T0 passes → export receipts for operator transfer. No secrets are used or printed anywhere. Branch: `citadel` (other branches are read-only inputs, never touched).

In [ ]:
# 0. Obtain repo at citadel (public clone, no credentials)
!test -d An-Ra-colab && echo HAVE_REPO || git clone --depth 50 -b citadel https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git An-Ra-colab
%cd An-Ra-colab
!git rev-parse HEAD && git log -1 --oneline

In [ ]:
# 1. Inspect the handover (authoritative next-action source)
!sed -n '1,80p' agent.md

In [ ]:
# 2. Inspect runtime first; install ONLY what is missing (each install counts toward the <10 GB budget)
!python -c "import torch; print('torch', torch.__version__)" 2>&1 | tail -1
!python -c "import torch_xla.version as xv; print('torch-xla', getattr(xv, '__version__', 'unknown'))" 2>&1 | tail -1
!python -c "import numpy; print('numpy', numpy.__version__)" 2>&1 | tail -1

In [ ]:
# 3. Minimal conditional install (edit ONLY if cell 2 showed a gap; keep versions compatible)
# !pip install -q torch torch-xla numpy 2>&1 | tail -2

In [ ]:
# 4. M0: environment probe (fail-closed; ABORT_NO_TPU on CPU fallback). Runtime TPU accelerator required.
from citadel_tpu import environment as env_mod
env = env_mod.main(out='docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json', require_tpu=True, platform_override='colab')
print({k: env[k] for k in ('platform','accelerator_detected','xla_device_count','torch_version','torch_xla_version','probe_pass')})

In [ ]:
# 5. T0: single-device one-update certification (MINI_SPEC, bucket 512, CE, one update)
from citadel_tpu import one_update
r0 = one_update.run(out='docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print({k: r0[k] for k in ('certification','loss','tokens_per_second','reload_identical')})

In [ ]:
# 6. STOP unless T0 passed. Export exact receipt files for operator transfer (no pasting).
assert r0.get('certification') == 'PASS', 'T0 did not pass — STOP. Diagnose, do not escalate.'
from google.colab import files
files.download('docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json')
files.download('docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print('exported; transfer these exact files back to the operator')